In [2]:
import glob
import os
import zipfile
from tqdm import tqdm

# ==========================================
# KONFIGURASI PATH
# ==========================================
ERA5_BASE_DIR = r"C:\Users\LENOVO\Documents\Tugaslah Pokoknya\Draft\Data ERA5"

print("Memulai proses pemeriksaan dan ekstraksi file...")

# Cari semua file di dalam folder ERA5 (termasuk subfolder tahun)
all_files = glob.glob(os.path.join(ERA5_BASE_DIR, "**", "*"), recursive=True)

extracted_count = 0
skipped_count = 0

for filepath in tqdm(all_files, desc="Mengekstrak File"):
  # Hindari folder, hanya proses file
  if os.path.isdir(filepath):
    continue

  # Cek apakah file merupakan ZIP (berdasarkan Magic Bytes 'PK\x03\x04')
  try:
    with open(filepath, "rb") as f:
      header = f.read(4)

    if header.startswith(b"PK\x03\x04"):
      target_dir = os.path.dirname(filepath)
      filename_without_ext = os.path.splitext(os.path.basename(filepath))[0]

      # Buat nama unik/subfolder jika diperlukan agar file tidak saling tertimpa
      with zipfile.ZipFile(filepath, "r") as z:
        for member in z.infolist():
          # Ekstrak file dan tambahkan prefix nama file asal agar tidak tertimpa
          extracted_filename = f"{filename_without_ext}_{member.filename}"
          extracted_path = os.path.join(target_dir, extracted_filename)

          # Ekstrak hanya jika belum ada
          if not os.path.exists(extracted_path):
            with (
                z.open(member) as source,
                open(extracted_path, "wb") as target,
            ):
              target.write(source.read())

      # Hapus atau ubah nama file .nc yang lama (yang berupa zip) agar tidak membingungkan
      # Di sini file zip asli diubah ekstensinya menjadi .zip
      if not filepath.endswith(".zip"):
        os.rename(filepath, filepath + ".zip")

      extracted_count += 1
    else:
      skipped_count += 1

  except Exception as e:
    print(f"\nGagal mengekstrak {filepath}: {e}")

print("\n=== PROSES EKSTRAKSI SELESAI ===")
print(f"Total file ZIP berhasil diekstrak : {extracted_count}")
print(f"Total file lain / non-ZIP          : {skipped_count}")
print("\nSekarang semua file NetCDF (.nc) asli sudah siap dibaca!")

Memulai proses pemeriksaan dan ekstraksi file...


Mengekstrak File: 100%|██████████| 489/489 [00:29<00:00, 16.62it/s]


=== PROSES EKSTRAKSI SELESAI ===
Total file ZIP berhasil diekstrak : 232
Total file lain / non-ZIP          : 232

Sekarang semua file NetCDF (.nc) asli sudah siap dibaca!


In [3]:
import glob
import os
import warnings
import pandas as pd
import xarray as xr
from tqdm import tqdm

# Mengabaikan warning internal dari xarray/pandas agar output terminal tetap bersih
warnings.filterwarnings("ignore")

# ==========================================
# 1. KONFIGURASI PATH FILE & FOLDER
# ==========================================
EXCEL_PATH = "Siklon Tropis.xlsx"
ERA5_BASE_DIR = r"C:\Users\LENOVO\Documents\Tugaslah Pokoknya\Draft\Data ERA5"
OUTPUT_PATH = "Siklon_Tropis_ERA5_Extracted.xlsx"

# ==========================================
# 2. BACA & PERSIAPKAN DATA SIKLON TROPIS
# ==========================================
print("Membaca data siklon dari Excel...")
df = pd.read_excel(EXCEL_PATH)

# Memastikan format kolom ISO_TIME berupa datetime
df["ISO_TIME"] = pd.to_datetime(df["ISO_TIME"])
df["YEAR"] = df["ISO_TIME"].dt.year

extracted_df_list = []

# ==========================================
# 3. PROSES EKSTRAKSI DATA ERA5 PER TAHUN
# ==========================================
print("\nMemulai proses ekstraksi variabel ERA5 berdasarkan LAT, LON, dan ISO_TIME...")

for year, group in df.groupby("YEAR"):
  year_dir = os.path.join(ERA5_BASE_DIR, str(year))

  # Cari semua file .nc hasil ekstraksi (mengabaikan file .zip asli)
  nc_files = [
      f
      for f in glob.glob(
          os.path.join(year_dir, "**", "*.nc"), recursive=True
      )
      if not f.endswith(".zip")
  ]

  if not nc_files:
    print(f"Peringatan: File .nc untuk tahun {year} tidak ditemukan.")
    extracted_df_list.append(group)
    continue

  print(
      f"Memproses tahun {year} ({len(group)} titik siklon) dari"
      f" {len(nc_files)} file NetCDF..."
  )

  try:
    # Membuka dataset NetCDF
    ds = xr.open_mfdataset(
        nc_files, combine="by_coords", combine_attrs="override"
    )

    # Deteksi otomatis nama variabel koordinat pada file NetCDF
    lat_name = next(
        (c for c in ["latitude", "lat", "Y"] if c in ds.coords), None
    )
    lon_name = next(
        (c for c in ["longitude", "lon", "X"] if c in ds.coords), None
    )
    time_name = next(
        (c for c in ["time", "valid_time", "valid_time0"] if c in ds.coords),
        None,
    )

    if not lat_name or not lon_name or not time_name:
      print(
          f"Gagal mendeteksi nama koordinat tahun {year}. Coords yang ada:"
          f" {list(ds.coords)}"
      )
      extracted_df_list.append(group)
      ds.close()
      continue

    # Penyesuaian rentang Bujur/Longitude jika berformat 0° - 360° menjadi -180° - 180°
    if float(ds[lon_name].max()) > 180:
      ds = ds.assign_coords({lon_name: (((ds[lon_name] + 180) % 360) - 180)})
      ds = ds.sortby(lon_name)

    var_names = list(ds.data_vars)
    extracted_rows = []

    # Iterasi setiap titik siklon tropis
    for idx, row in tqdm(
        group.iterrows(), total=len(group), desc=f"Tahun {year}"
    ):
      row_dict = row.to_dict()
      try:
        # Ekstraksi berbasis pencocokan Nearest Neighbor (Spasial & Temporal)
        sub_ds = ds.sel(
            {
                time_name: row["ISO_TIME"],
                lat_name: row["LAT"],
                lon_name: row["LON"],
            },
            method="nearest",
        )

        for var in var_names:
          val = sub_ds[var].values
          row_dict[f"era5_{var}"] = (
              float(val.flatten()[0]) if val.size > 1 else float(val)
          )

      except Exception:
        for var in var_names:
          row_dict[f"era5_{var}"] = None

      extracted_rows.append(row_dict)

    extracted_df_list.append(pd.DataFrame(extracted_rows))
    ds.close()

  except Exception as err:
    print(f"Gagal memproses NetCDF tahun {year}: {err}")
    extracted_df_list.append(group)

# ==========================================
# 4. GABUNGKAN DATA & SIMPAN KE EXCEL
# ==========================================
final_df = pd.concat(extracted_df_list, ignore_index=True)
final_df.drop(columns=["YEAR"], inplace=True, errors="ignore")

final_df.to_excel(OUTPUT_PATH, index=False)
print(f"\nProses Selesai! File hasil ekstraksi disimpan di: {OUTPUT_PATH}")

Membaca data siklon dari Excel...

Memulai proses ekstraksi variabel ERA5 berdasarkan LAT, LON, dan ISO_TIME...
Memproses tahun 2001 (1036 titik siklon) dari 24 file NetCDF...


Tahun 2001: 100%|██████████| 1036/1036 [00:32<00:00, 31.53it/s]


Memproses tahun 2002 (1198 titik siklon) dari 36 file NetCDF...


Tahun 2002: 100%|██████████| 1198/1198 [00:55<00:00, 21.43it/s]


Memproses tahun 2003 (1564 titik siklon) dari 33 file NetCDF...


...

Memproses tahun 2024 (940 titik siklon) dari 18 file NetCDF...


Tahun 2024: 100%|██████████| 940/940 [01:12<00:00, 13.04it/s]


Memproses tahun 2025 (924 titik siklon) dari 21 file NetCDF...


Tahun 2025: 100%|██████████| 924/924 [01:25<00:00, 10.83it/s]



Proses Selesai! File hasil ekstraksi disimpan di: Siklon_Tropis_ERA5_Extracted.xlsx
